# 4.5 Entrenamiento final

En esta sección entrenamos $R$ semillas de cada modelo con la configuración seleccionada y verificamos que cada corrida esté completa, que el conjunto de prueba se haya evaluado una sola vez y que la configuración sea la misma en todas las semillas de un modelo.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from src import config as C
from src import plots as P
from src.persist import is_done, load_pickle, save_pickle

P.setup()
C.ensure_dirs()
pd.set_option("display.precision", 4)
print("Resultados:", C.RESULTS.name, "| modelos:", ", ".join(C.MODELS))

Resultados: results | modelos: CNN, MobileNetV2, ResNet-18


In [2]:
RUN_TRAINING = False     # True solo si se quiere entrenar desde aquí; lo recomendado es la terminal

if RUN_TRAINING:
    import subprocess
    # Se lanza como proceso aparte: así el DataLoader con workers funciona en Windows y
    # una corrida interrumpida se reanuda igual que desde la terminal.
    subprocess.run([sys.executable, "-m", "src.runner", "final"], cwd=ROOT, check=False)

## Protocolo de cada corrida

1. Se fija la semilla y se inicializa el modelo desde cero.
2. Se entrena con *early stopping* sobre el F1 macro de validación (paciencia de 15 épocas, mejora mínima de 1e-4, máximo 100 épocas) y se restaura el mejor estado.
3. Al final de cada época se guardan la historia y un punto de control completo; el mejor modelo se guarda cada vez que mejora la validación.
4. Con el mejor modelo se predicen validación y prueba. El conjunto de prueba se evalúa una sola vez por corrida.
5. Se guardan las métricas y los tiempos de inferencia.

Las corridas se ejecutaron de forma intercalada (semilla 0 de los tres modelos, luego semilla 1, y así sucesivamente), de modo que en todo momento los tres modelos tenían el mismo número de semillas terminadas.

In [3]:
REQUERIDOS = ["config.pkl", "config.json", "env.pkl", "model_info.pkl", "history.pkl", "checkpoint_last.pt",
              "best_model.pt", "val_predictions.pkl", "test_predictions.pkl", "metrics_test.pkl", "timing.pkl", "DONE"]
pa = load_pickle(C.COMMON / "power_analysis.pkl")
R = pa["R_final"]
filas = []
for m in C.MODELS:
    for s in range(R):
        d = C.run_dir(m, s)
        fila = {"modelo": m, "semilla": s, "completa": is_done(d),
                "faltan": ", ".join(f for f in REQUERIDOS if not (d / f).exists()) or "-"}
        if is_done(d):
            t = load_pickle(d / "timing.pkl")
            fila.update({"épocas": t["epochs_run"], "mejor época": t["best_epoch"], "early stopping": t["stopped_early"],
                         "F1 val": round(t["best_val_f1_macro"], 4), "min de entrenamiento": round(t["train_time_s"] / 60, 1)})
        filas.append(fila)
estado = pd.DataFrame(filas)
save_pickle(estado, C.SUMMARY / "estado_corridas.pkl")
display(estado)
assert estado["completa"].all(), "Hay corridas sin terminar: python -m src.runner final"

,modelo,semilla,completa,faltan,épocas,mejor época,early stopping,F1 val,min de entrenamiento
0,CNN,0,True,-,100,96,False,0.8627,8.9
1,CNN,1,True,-,95,79,True,0.8431,8.4
2,CNN,2,True,-,100,99,False,0.8635,8.9
3,CNN,3,True,-,100,87,False,0.8583,8.9
4,CNN,4,True,-,100,91,False,0.8626,8.9
5,CNN,5,True,-,61,45,True,0.8095,5.4
6,CNN,6,True,-,100,88,False,0.8431,8.9
7,CNN,7,True,-,100,94,False,0.8614,8.8
8,MobileNetV2,0,True,-,87,71,True,0.8388,7.7
9,MobileNetV2,1,True,-,100,93,False,0.8325,8.9


## Verificaciones

In [4]:
import pickle

chequeos = {}
cfgs = {m: [load_pickle(C.run_dir(m, s) / "config.pkl") for s in range(R)] for m in C.MODELS}
chequeos["misma configuración en todas las semillas de cada modelo"] = all(
    all(c["hp"] == cs[0]["hp"] for c in cs) for cs in cfgs.values())
comunes = ["batch_size", "max_epochs", "patience", "min_delta", "warmup_epochs", "eta_min_factor", "grad_clip", "amp"]
chequeos["mismo presupuesto y protocolo entre modelos"] = all(
    cfgs[m][0]["hp"][k] == cfgs[C.MODELS[0]][0]["hp"][k] for m in C.MODELS for k in comunes)
chequeos["misma partición (huella del test) en todas las corridas"] = len({c["test_fingerprint"] for cs in cfgs.values() for c in cs}) == 1
chequeos["régimen desde cero en todas las corridas"] = all(c["regime"].startswith("desde cero") for cs in cfgs.values() for c in cs)
pred = {(m, s): load_pickle(C.run_dir(m, s) / "test_predictions.pkl") for m in C.MODELS for s in range(R)}
chequeos["mismas imágenes de prueba y en el mismo orden"] = len({tuple(p["filepaths"]) for p in pred.values()}) == 1
chequeos["test evaluado antes de cerrar la corrida (una sola vez)"] = all(
    (C.run_dir(m, s) / "test_predictions.pkl").stat().st_mtime <= (C.run_dir(m, s) / "DONE").stat().st_mtime for m, s in pred)

def tipos_neutros(o):
    if isinstance(o, dict):
        return all(tipos_neutros(v) for v in o.values())
    if isinstance(o, (list, tuple)):
        return all(tipos_neutros(v) for v in o)
    return isinstance(o, (str, int, float, bool, type(None), np.ndarray, np.generic, pd.DataFrame, pd.Series))

pkls = [f for m in C.MODELS for s in range(R) for f in C.run_dir(m, s).glob("*.pkl")]
chequeos[f"los {len(pkls)} .pkl de las corridas se leen sin GPU ni código del modelo"] = all(
    tipos_neutros(pickle.load(open(f, "rb"))) for f in pkls)
display(pd.Series(chequeos).to_frame("cumple"))
assert all(chequeos.values())

,cumple
misma configuración en todas las semillas de cada modelo,True
mismo presupuesto y protocolo entre modelos,True
misma partición (huella del test) en todas las corridas,True
régimen desde cero en todas las corridas,True
mismas imágenes de prueba y en el mismo orden,True
test evaluado antes de cerrar la corrida (una sola vez),True
los 216 .pkl de las corridas se leen sin GPU ni código del modelo,True


In [5]:
env = load_pickle(C.run_dir(C.MODELS[0], 0) / "env.pkl")
total_h = estado["min de entrenamiento"].sum() / 60
display(Markdown(f"Realizamos {len(estado)} corridas ({R} semillas × 3 modelos) en {env['gpu'] or 'CPU'} "
                 f"(PyTorch {env['torch']}, CUDA {env['cuda']}, cuDNN {env['cudnn']}). El tiempo total de entrenamiento "
                 f"de las corridas finales fue de {total_h:.1f} h."))

Realizamos 24 corridas (8 semillas × 3 modelos) en NVIDIA GeForce RTX 4060 Laptop GPU (PyTorch 2.15.0+cu132, CUDA 13.2, cuDNN 92600). El tiempo total de entrenamiento de las corridas finales fue de 3.4 h.

## Conclusión

Todas las corridas terminaron con sus archivos completos. Dentro de cada modelo las semillas comparten la configuración, y entre modelos el protocolo y el presupuesto son iguales. El conjunto de prueba es el mismo, en el mismo orden, en todas las corridas, y se evaluó una sola vez por corrida con los hiperparámetros ya fijados.